# 03 — NLP Pathology Label ImputationRegex + ConText negation detection to impute missing `Pathology.Categories` from free-text notes.

In [ ]:
import pandas as pdfrom gtex_biomarkers.config import Configfrom gtex_biomarkers.data import load_raw_datafrom gtex_biomarkers.labels import extract_categories, CATEGORY_PATTERNSConfig.ensure_dirs()_, _, _, df_meta_url = load_raw_data()

## Liver Pathology — Most Common Raw Strings

In [ ]:
liver_all = df_meta_url[df_meta_url["Tissue"].astype(str) == "Liver"].copy()liver_all["Pathology.Categories"].fillna("NA").value_counts()

## Validate on Ground-Truth Rows

In [ ]:
# Apply extraction to rows where categories already existliver_gt = liver_all[liver_all["Pathology.Categories"].notna()].copy()liver_gt["pred_list"] = liver_gt["Pathology.Notes"].apply(extract_categories)print(f"Ground-truth rows: {len(liver_gt)}")display(liver_gt[["Pathology.Categories", "Pathology.Notes", "pred_list"]].head(10))

## Apply to Missing Rows & Save

In [ ]:
liver_missing = liver_all[liver_all["Pathology.Categories"].isna()].copy()print(f"Rows with missing categories: {len(liver_missing)}")liver_missing["pred_list"] = liver_missing["Pathology.Notes"].apply(extract_categories)liver_missing["Pathology.Categories.Imputed"] = liver_missing["pred_list"].apply(    lambda lst: ", ".join(lst) if lst else float("nan"))filled = liver_missing["Pathology.Categories.Imputed"].notna().sum()print(f"Imputed: {filled} / {len(liver_missing)}")# Combine original + imputedliver_all["Pathology.Categories.Final"] = liver_all["Pathology.Categories"].copy()imputed_mask = liver_all["Pathology.Categories"].isna()liver_all.loc[imputed_mask, "Pathology.Categories.Final"] = (    liver_missing["Pathology.Categories.Imputed"])liver_all["SUBJID"] = liver_all["Tissue.Sample.ID"].astype(str).str.split("-").str[:2].str.join("-")liver_all.to_csv(Config.PROCESSED_DIR / "liver_pathology_labels_imputed.csv", index=False)print(f"Saved → {Config.PROCESSED_DIR / 'liver_pathology_labels_imputed.csv'}")